# 🧠 Citizen Science Project Analysis

This notebook processes and analyzes citizen science projects using LangChain and LLM analysis.

## 📦 Data Preparation

To use this tool, the data needs to be prepared in a specific format. The preferred methods for obtaining the data are:

1.  **Using an available API:** If there is an API that provides access to the project database, this is the most efficient method.
2.  **Downloading the entire database:** If the complete database can be downloaded directly, this method is also suitable.
3.  **Web Scraping:** If neither an API nor a direct download is available, the project titles and descriptions can be obtained by scraping the relevant website(s).

 ## 📁 file and directory instructions
Regardless of the method used to obtain the data, the final input should be a CSV file named `database.csv` with at least two columns:

*   A column containing the **title** of each project.
*   A column containing a **description or summary** of each project. The column name for the summary should ideally be `summary_en`.

Ensure the `database.csv` file is placed in the `/content/` directory (found on the left side of the screen under the icon "files") or update the file path in the notebook accordingly.

# 🧰 Setup of libraries and tools
**Purpose:** Installs the `langchain_openai` library.
**Relation:** This is the initial setup step, ensuring that the necessary library for interacting with the language model via LangChain is available. It precedes all code that uses LangChain components with the OpenRouter API.

In [ ]:
!pip install langchain_openai

**Purpose:** Imports essential Python libraries.
**Relation:** This cell provides access to the tools needed throughout the notebook: pandas for data handling (used in data loading and processing), LangChain modules for building and running the LLM chain, Pydantic for defining the output structure, and `os`/`time` for environment variable access and rate limiting. This cell must be executed before any code that utilizes these libraries.

In [ ]:
# Import required libraries
import pandas as pd
from langchain.prompts import PromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAI
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, SecretStr
from typing import Optional
import os
import time

**Purpose:** Defines the Pydantic model for the LLM's output structure.

**Relation:** This `ProjectAnalysis` class sets the expected format for the data returned by the language model after analyzing a project. It is directly used by the `PydanticOutputParser` in a later cell to ensure the LLM's output conforms to this structure, which is crucial for subsequent data processing and analysis steps.

In [ ]:
# Define the output schema using Pydantic
class ProjectAnalysis(BaseModel):
    """Analysis result for a citizen science project."""
    is_citizen_science: int = Field(description="Probability that this is a citizen science project (1 to 5) where 5 is certainly a citizen science project")
    project_name: str = Field(description="Name of the project")
    row: int = Field(description="Row number in the original DataFrame")

**Purpose:** Loads the project data from a CSV file.

**Relation:** This cell reads the primary dataset (`database.csv`) into a pandas DataFrame (`df`). This DataFrame serves as the input for the LLM processing chain defined later. The subsequent steps of analyzing and filtering projects rely on the data loaded here. A copy (`result_df`) is also created, intended for storing results alongside the original data.

In [ ]:
# Load the aggregated data
print('Starting data processing with LangChain...')

# Attempt to read the CSV with a different engine for better handling of complex formatting
df = pd.read_csv('/content/database.csv', engine='python')

# Create a copy of the dataframe for storing results
result_df = df.copy()

# Display basic info about the dataset
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

# 🔑API key configuration

**Purpose:** Checks for the presence of the OpenRouter API key.

**Relation:** This cell is a prerequisite check before initializing the language model that uses the OpenRouter API. It ensures that the necessary authentication credential (`OPENROUTER_API_KEY`) is available, preventing errors in the subsequent cell where the `ChatOpenRouter` is initialized.

In [ ]:
# Initialize the LLM using OpenRouter API key
# Note: Make sure .env file contains OPENROUTER_API_KEY

api_key = "PUT YOUR API KEY HERE"
if not api_key:
    print("Warning: OPENROUTER_API_KEY not found in environment variables")
    print("Make sure your .env file contains: OPENROUTER_API_KEY=your-api-key-here")
else:
    print("✓ API key found in environment variables")
    print(f"API key starts with: {api_key[:10]}...")

**Purpose:** Defines a custom LangChain Chat Model for OpenRouter.
**Relation:** This `ChatOpenRouter` class is a crucial component for integrating LangChain with the OpenRouter API. It extends LangChain's `ChatOpenAI` and configures the base URL and API key to point to OpenRouter's service. This class is instantiated in the next cell to create the `llm` object used in the processing chain.

In [ ]:
# Define custom ChatOpenRouter class for OpenRouter API
class ChatOpenRouter(ChatOpenAI):
    openai_api_key: Optional[SecretStr] = Field(
        alias="api_key",
        default_factory=lambda: None,
    )
    @property
    def lc_secrets(self) -> dict[str, str]:
        return {"openai_api_key": "OPENROUTER_API_KEY"}

    def __init__(self,
                openai_api_key: Optional[str] = None,
                **kwargs):
        openai_api_key = (
           "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"
        )
        super().__init__(
            base_url="https://openrouter.ai/api/v1",
            api_key=SecretStr(openai_api_key) if openai_api_key else None,
            **kwargs
        )

# 🧱Model initialization
**Purpose:** Initializes the LLM, output parser, and prompt template.
**Relation:** This cell brings together several key components: it instantiates the `ChatOpenRouter` (defined in the previous cell) as the language model (`llm`), creates a `PydanticOutputParser` based on the `ProjectAnalysis` model (defined earlier) to handle the LLM's output, and defines the `PromptTemplate` that guides the LLM's analysis. These three elements are combined in the subsequent cell to form the processing chain.

**If the definition of citizen science changes or more specific instruction needs to be provided to the model this cell is the one that needs to be modifidied**. Specifically the part that starts with 'Evaluate if this is a citizen science project based on the title and summary. Consider the following criteria.....'

In [ ]:
# Initialize the LLM
llm = ChatOpenRouter(
    model_name="google/gemini-2.5-flash"
)

# Create output parser for structured output
output_parser = PydanticOutputParser(pydantic_object=ProjectAnalysis)

# Create a prompt template with placeholders
prompt_template = PromptTemplate(
    input_variables=["title", "summary"],
    template="""
    Analyze the following project:

    Title: {title}
    Summary: {summary}
    row : {row}

    Evaluate if this is a citizen science project based on the title and summary.
    Consider the following criteria for citizen science:
    - Citizen science projects actively involve citizens in scientific endeavour that generates new
    knowledge or understanding. Citizens may act as contributors, collaborators, or as project
    leader and have a meaningful role in the project.
    - Both the professional scientists and the citizen scientists benefit from taking part. Benefits
    may include the publication of research outputs, learning opportunities, personal enjoyment,
    social benefits, satisfaction through contributing to scientific evidence e.g. to address local,
    national and international issues, and through that, the potential to influence policy.
    - Citizens involved during the project must be alive at the time of actively contributing towards the project.

    Provide a json response

    {format_instructions}
    """,
    partial_variables={"format_instructions": output_parser.get_format_instructions()}
)

print("Prompt template created successfully")

#⚙️ processing chain setup

**Purpose:** Creates the LangChain processing chain.

**Relation:** This cell defines the sequence of operations for processing each project. It links the `prompt_template`, `llm`, and `StrOutputParser` (created in previous cells) using the `|` operator, forming a `Runnable` chain. This chain is invoked in the next cell for each row of the DataFrame.

In [ ]:
# Create the processing chain
chain = (
    prompt_template
    | llm
    | StrOutputParser()
)

print("Processing chain created successfully")

#🔁 iteration / project analysis loop
**Purpose:** Processes each row of the DataFrame using the LangChain chain.
**Relation:** This cell iterates through the `df` DataFrame (loaded earlier) and applies the `chain` (created in the previous cell) to each project's title and summary. The results of the LLM analysis are collected in the `processed_results` list. This is the core processing loop that generates the initial analysis data.

In [ ]:
# Process each row in the dataframe
processed_results = []
i = 0


for index, row in df.iterrows():
    if not i % 3000 == 0:
         i += 1
         continue  # Skip rows that are not multiples of 3000

    try:
        print(f"Processing row {index}/{len(df)}: {row['title'][:50]}...")

        # Prepare input for the chain
        input_data = {
            "title": row['title'],
            "summary": row['summary_en'],
            "row" : index # Using English summary
        }

        # Process with the actual LangChain
        if api_key:
            result = chain.invoke(input_data)
        else:
            result = "Error: No API key provided"

        processed_results.append(result)
        i += 1
        time.sleep(1)  # Sleep to avoid hitting API rate limits

    except Exception as e:
        print(f"Error processing row {index}: {e}")
        processed_results.append("Error in processing")

print(f"\nProcessed {len(processed_results)} results")

**Purpose:** Saves the raw processed results to a JSON file.
**Relation:** This cell takes the `processed_results` list (generated in the previous cell) and writes it to a JSON file. This serves as an intermediate save point, allowing the analysis results to be persistent and used by the subsequent sorting and filtering steps without needing to re-run the LLM processing.

In [ ]:
# Save processed results to outputs folder
import json
from datetime import datetime

# Create outputs directory if it doesn't exist
os.makedirs('../outputs', exist_ok=True)

# Generate timestamp for filename
filename = f'../content/result_test.json'

# Save results as JSON
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(processed_results, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {filename}")
print(f"Number of results saved: {len(processed_results)}")

**Purpose:** Loads, sorts, and saves the initial processed results.
**Relation:** This cell reads the JSON file saved by the previous cell, extracts and validates the JSON data from the LLM's output, sorts the projects based on their `is_citizen_science` score, and saves the sorted results to a CSV file (`sorted_result_test.csv`). It also displays a summary of the scores and makes the sorted data available as `sorted_df` and `sorted_results` for the next filtering step.

In [ ]:
# Citizen Science Projects Sorting - Complete Analysis
# ====================================================
# This cell contains all the functionality to sort citizen science projects by their scores.
# Modify the input_file path below to point to your JSON file.

# Import Required Libraries
import json
import pandas as pd
import sys
from pathlib import Path
from typing import List, Dict, Any, Optional
import re

# Add the scripts directory to the path to import functions
sys.path.append('./scripts')

# Define helper functions (inline definitions)
def extract_json_from_string(json_string: str) -> Optional[Dict[str, Any]]:
    """Extract JSON data from a string that contains a JSON code block."""
    clean_json = re.sub(r'^```json\s*', '', json_string.strip())
    clean_json = re.sub(r'\s*```$', '', clean_json)

    try:
        return json.loads(clean_json)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        print(f"Problematic string: {json_string[:100]}...")
        return None

def load_results(file_path: str) -> List[Dict[str, Any]]:
    """Load results from a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        results = []
        if isinstance(raw_data, list):
            for item in raw_data:
                if isinstance(item, str):
                    extracted_data = extract_json_from_string(item)
                    if extracted_data:
                        results.append(extracted_data)
                elif isinstance(item, dict):
                    results.append(item)
            return results
        else:
            print(f"Warning: Expected a list, got {type(raw_data)}. Wrapping in list.")
            return [raw_data]

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []

def sort_by_citizen_science_score(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Sort results by citizen science score in descending order."""
    return sorted(results, key=lambda x: x.get('is_citizen_science', 0), reverse=True)

def save_sorted_results(results: List[Dict[str, Any]], output_path: str):
    """Save sorted results to a CSV file."""
    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Sorted results saved to: {output_path}")

def display_score_summary(results: List[Dict[str, Any]]):
    """Display a summary of the score distribution."""
    print("\n" + "="*50)
    print("CITIZEN SCIENCE SCORE DISTRIBUTION")
    print("="*50)

    # Count by score
    score_counts = {}
    for result in results:
        score = result.get('is_citizen_science', 0)
        score_counts[score] = score_counts.get(score, 0) + 1

    # Display distribution
    for score in sorted(score_counts.keys(), reverse=True):
        count = score_counts[score]
        percentage = (count / len(results)) * 100 if results else 0
        print(f"Score {score}: {count:4d} projects ({percentage:5.1f}%)")

    print(f"\nTotal projects: {len(results)}")

    # Show top 10 projects
    top_projects = results[:10]
    if top_projects:
        print(f"\nTOP 10 CITIZEN SCIENCE PROJECTS:")
        print("-" * 60)
        for i, project in enumerate(top_projects, 1):
            score = project.get('is_citizen_science', 0)
            name = project.get('project_name', 'Unknown')
            print(f"{i:2d}. [{score}] {name}")

# ====================================================
# CONFIGURATION - MODIFY THIS SECTION
# ====================================================

# SET YOUR INPUT FILE PATH HERE
# Option 1: Absolute path
# input_file = '/home/marcello/projects/csnl/CS.nl/data/processed/your_results.json'

# Option 2: Relative path (relative to notebook location)
input_file = '/content/result_test.json'

# Optional: Set custom output path (if not specified, it will be auto-generated)
output_file = '/content/sorted_result_test.csv'  # Set to None for auto-generation, or specify a path like './outputs/sorted_results.csv'

# ====================================================
# MAIN EXECUTION
# ====================================================

print("CITIZEN SCIENCE PROJECT SORTING ANALYSIS")
print("="*50)
print(f"Input file: {input_file}")
print(f"Output file: {output_file if output_file else 'Will be auto-generated'}")
print()

# Check if input file exists
if not Path(input_file).exists():
    print(f"❌ Error: Input file '{input_file}' not found!")
    print(f"Please check the path and make sure the file exists.")
    print(f"Current working directory: {Path.cwd()}")
    print(f"Available files in current directory: {list(Path('.').glob('*.json'))}")
else:
    print(f"✅ Input file found: {input_file}")

    # Load results from the JSON file
    print("Loading results...")
    results = load_results(str(input_file))

    if not results:
        print("❌ No results found in the input file!")
    else:
        print(f"✅ Loaded {len(results)} projects")

        # Sort by citizen science score
        print("Sorting by citizen science score...")
        sorted_results = sort_by_citizen_science_score(results)
        print(f"✅ Sorting completed!")

        # Determine output path
        if output_file:
            output_path = output_file
        else:
            input_path = Path(input_file)
            output_path = input_path.parent / f"{input_path.stem}_sorted.csv"

        print(f"Saving results to: {output_path}")

        # Save sorted results to CSV
        save_sorted_results(sorted_results, str(output_path))

        # Display summary
        display_score_summary(sorted_results)

        # Optional: Create DataFrame for further analysis
        df = pd.DataFrame(sorted_results)
        print("\n" + "="*50)
        print("DATASET OVERVIEW")
        print("="*50)
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")

        # Show score statistics if available
        if 'is_citizen_science' in df.columns:
            print(f"\nScore Statistics:")
            print(f"Mean score: {df['is_citizen_science'].mean():.2f}")
            print(f"Median score: {df['is_citizen_science'].median():.2f}")
            print(f"Max score: {df['is_citizen_science'].max()}")
            print(f"Min score: {df['is_citizen_science'].min()}")

        print(f"\n🎉 Analysis completed successfully!")
        print(f"📊 Sorted {len(sorted_results)} projects by citizen science score")
        print(f"💾 Results saved to: {output_path}")

        # Display first few rows
        print("\nFirst 5 rows of sorted data:")
        display(df.head())

        # Make the DataFrame available for further use
        globals()['sorted_df'] = df
        globals()['sorted_results'] = sorted_results

        print("\n📌 Variables created:")
        print("   - sorted_df: DataFrame with sorted results")
        print("   - sorted_results: List of sorted dictionaries")

**Purpose:** Filters projects by score and matches with original data.
**Relation:** This cell takes the sorted results (from the previous cell's output CSV) and the original `database.csv` (loaded earlier). It filters the sorted results to include only projects with a citizen science score of 4 or 5. It then uses the row numbers to match these filtered projects with their corresponding entries in the original `database.csv`, creating a new DataFrame `matched_df` (also stored as `filtered_citizen_science_df`). This cell provides the final filtered dataset used for further analysis or reporting.

In [ ]:
# Second Filtration - Citizen Science Projects Filter
# Filter citizen science projects with scores 4 or 5 and match with whole.csv data

# Import Required Libraries
import pandas as pd
import os
from datetime import datetime

# Define Filtration Function
def filter_citizen_science_projects(processed_results_path, whole_csv_path, output_path=None):
    """
    Filter citizen science projects with scores 4 or 5 and match with whole.csv data.

    Args:
        processed_results_path (str): Path to the processed results CSV file
        whole_csv_path (str): Path to the whole.csv file
        output_path (str, optional): Path for output file. If None, generates timestamped filename.

    Returns:
        pd.DataFrame: Filtered dataframe with matched projects
    """

    print(f"Reading processed results from: {processed_results_path}")
    try:
        # Read the processed results file
        processed_df = pd.read_csv(processed_results_path)
        print(f"Loaded {len(processed_df)} rows from processed results")
    except Exception as e:
        print(f"Error reading processed results file: {e}")
        return None

    print(f"Reading whole.csv from: {whole_csv_path}")
    try:
        # Read the whole.csv file
        # Make sure to read with the correct index if needed
        whole_df = pd.read_csv(whole_csv_path)
        print(f"Loaded {len(whole_df)} rows from whole.csv")
    except Exception as e:
        print(f"Error reading whole.csv file: {e}")
        return None

    # Filter for citizen science scores of 4 or 5
    print("Filtering for citizen science scores of 4.0 or 5.0...")
    filtered_processed = processed_df[
        (processed_df['is_citizen_science'] == 4.0) |
        (processed_df['is_citizen_science'] == 5.0)
    ].copy()

    print(f"Found {len(filtered_processed)} projects with citizen science scores of 4 or 5")

    if len(filtered_processed) == 0:
        print("No projects found with the specified criteria")
        return None

    # Get the row numbers for matching from the 'row' column
    row_numbers_to_match = filtered_processed['row'].tolist()
    print(f"Row numbers to match: {len(row_numbers_to_match)} rows")

    # Match with whole.csv using the DataFrame index
    print("Matching with whole.csv data using DataFrame index...")
    # Since the 'row' column in processed_df corresponds to the original index,
    # we filter whole_df based on its index using the list of row numbers
    matched_df = whole_df[whole_df.index.isin(row_numbers_to_match)].copy()


    print(f"Successfully matched {len(matched_df)} projects")

    # Sort by 'No' column to maintain order
    # Assuming the original order is important and corresponds to the index
    matched_df = matched_df.sort_index().reset_index(drop=False)
    matched_df = matched_df.rename(columns={'index': 'No'}) # Rename the index column back to 'No'

    # Generate output filename if not provided
    if output_path is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_path = f"outputs/filtered_citizen_science_{timestamp}.csv"

    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Save the filtered results
    print(f"Saving results to: {output_path}")
    matched_df.to_csv(output_path, index=False)

    # Print summary statistics
    print("\n" + "="*60)
    print("FILTRATION SUMMARY")
    print("="*60)
    print(f"Total projects in processed results: {len(processed_df)}")
    count_4 = len(processed_df[processed_df['is_citizen_science'] == 4.0])
    count_5 = len(processed_df[processed_df['is_citizen_science'] == 5.0])
    print(f"Projects with score 4.0: {count_4}")
    print(f"Projects with score 5.0: {count_5}")
    print(f"Total filtered projects: {len(filtered_processed)}")
    print(f"Successfully matched projects: {len(matched_df)}")
    print(f"Output saved to: {output_path}")
    print("="*60)

    return matched_df

# Set File Paths - MODIFY THESE PATHS AS NEEDED
processed_results_path = "/content/sorted_result_test.csv"
whole_csv_path = "/content/database.csv"
output_path = "/content/four_and_fives.csv"  # Will generate timestamped filename automatically

# Alternative: Set specific output path
# output_path = "outputs/my_filtered_results.csv"

# Check if files exist
if not os.path.exists(processed_results_path):
    print(f"Error: Processed results file not found at {processed_results_path}")
else:
    if not os.path.exists(whole_csv_path):
        print(f"Error: whole.csv file not found at {whole_csv_path}")
    else:
        # Run Filtration and Display Results
        result_df = filter_citizen_science_projects(processed_results_path, whole_csv_path, output_path)

        if result_df is not None:
            print("\nFirst 5 rows of filtered results:")
            print(result_df.head())

            # Store result in a variable for further use in the notebook
            filtered_citizen_science_df = result_df
            print(f"\nFiltered dataframe stored in variable: filtered_citizen_science_df")
            print(f"Shape: {filtered_citizen_science_df.shape}")
        else:
            print("Filtration failed or no results found")

**Purpose:** Loads the filtered citizen science projects data from a CSV file.

**Relation:** This cell reads the CSV file generated by the previous filtering step (`four_and_fives.csv`) into a pandas DataFrame (`df1`). This DataFrame contains the projects that were identified as likely citizen science projects (scores 4 or 5). It also creates a copy (`result_df1`) for potential future use. This cell is a prerequisite for the subsequent processing steps that will further analyze this filtered dataset.

In [ ]:
# Load the aggregated data
print('Starting data processing with LangChain...')

df1 = pd.read_csv('/content/four_and_fives.csv', engine='python')

# Create a copy of the dataframe for storing results
result_df1 = df1.copy()

# Display basic info about the dataset
print(f"\nDataset shape: {df1.shape}")
print(f"Columns: {list(df1.columns)}")
df1.head()

**Purpose:** Re-initializes the LLM, output parser, and prompt template for the second analysis phase.

**Relation:** This cell sets up the necessary components for analyzing the filtered dataset. It re-instantiates the `ChatOpenRouter` with the specified model, creates a `PydanticOutputParser` based on the `ProjectAnalysis` model, and defines a new `PromptTemplate`. The template is specifically designed for the second analysis, using a refined set of criteria for identifying citizen science projects. These components will be used together in the next cell to form the processing chain for the filtered data.

In [ ]:
# Initialize the LLM
llm = ChatOpenRouter(
    model_name="google/gemini-2.5-flash"
)

# Create output parser for structured output
output_parser = PydanticOutputParser(pydantic_object=ProjectAnalysis)

# Create a prompt template with placeholders
prompt_template = PromptTemplate(
    input_variables=["title", "summary"],
    template="""
    You are an expert in evaluating research projects for their alignment with citizen science principles.
    Analyze the following project and determine whether it qualifies as a *citizen science* project.


    Title: {title}
    Summary: {summary}
    row : {row}

   1. Citizens are actively involved **during the execution** of the project — as contributors, collaborators, or leaders — and play a **meaningful role**  that generates new
    knowledge or understanding.
    2. The involvement results in **mutual benefits**: scientific knowledge generation **and** value for the citizens (e.g., learning, enjoyment, social impact, or policy influence).
    3. Citizens involved must be **alive during the project’s active phase**.
    Provide a json response

    {format_instructions}
    """,
    partial_variables={"format_instructions": output_parser.get_format_instructions()}
)

print("Prompt template created successfully")

**Purpose:** Processes each row of the filtered DataFrame using the LangChain chain.

**Relation:** This cell iterates through the `df1` DataFrame (loaded in the previous cell, containing the filtered projects) and applies the `chain` (which was defined earlier and uses the updated prompt template and LLM) to each project's title and summary. The results of this second LLM analysis are collected in the `processed_results` list. This loop performs the core analysis on the subset of projects identified in the first filtration step.

In [ ]:
# Process each row in the dataframe
processed_results = []
i = 0


for index, row in df1.iterrows():
    #if not i % 15 == 0:
      #  i += 1
        #continue  # Skip rows that are not multiples of 1346

    try:
        print(f"Processing row {index}/{len(df1)}: {row['title'][:50]}...")

        # Prepare input for the chain
        input_data = {
            "title": row['title'],
            "summary": row['summary_en'],
            "row" : index # Using English summary
        }

        # Process with the actual LangChain
        if api_key:
            result = chain.invoke(input_data)
        else:
            result = "Error: No API key provided"

        processed_results.append(result)
        i += 1
        time.sleep(1)  # Sleep to avoid hitting API rate limits

    except Exception as e:
        print(f"Error processing row {index}: {e}")
        processed_results.append("Error in processing")

print(f"\nProcessed {len(processed_results)} results")

**Purpose:** Saves the raw processed results from the second analysis to a JSON file.

**Relation:** This cell takes the `processed_results` list generated from processing the filtered DataFrame and writes it to a JSON file (`second_filtration.json`). This action preserves the results of the second LLM analysis, making them available for subsequent sorting and filtering steps without needing to re-run the LLM processing.

In [ ]:
# Save processed results to outputs folder
import json
from datetime import datetime

# Create outputs directory if it doesn't exist
os.makedirs('../outputs', exist_ok=True)

# Generate timestamp for filename
filename = f'../content/second_filtration.json'

# Save results as JSON
with open(filename, 'w', encoding='utf-8') as f:
    json.dump(processed_results, f, indent=2, ensure_ascii=False)

print(f"Results saved to: {filename}")
print(f"Number of results saved: {len(processed_results)}")

#📊 analysis output or result summary
**Purpose:** Loads, sorts, and saves the results from the second filtration step.

**Relation:** This cell reads the JSON file (`second_filtration.json`) created in the previous step, extracts and validates the JSON data, sorts the projects based on their `is_citizen_science` score from the second analysis, and saves the sorted results to a CSV file (`sorted_second_filtration.csv`). It also displays a summary of the scores and makes the sorted data available as `sorted_df` and `sorted_results` for potential further use. This cell concludes the sorting and initial analysis phase of the second filtration.

In [ ]:
# Citizen Science Projects Sorting - Complete Analysis
# ====================================================
# This cell contains all the functionality to sort citizen science projects by their scores.
# Modify the input_file path below to point to your JSON file.

# Import Required Libraries
import json
import pandas as pd
import sys
from pathlib import Path
from typing import List, Dict, Any, Optional
import re

# Add the scripts directory to the path to import functions
sys.path.append('./scripts')

# Define helper functions (inline definitions)
def extract_json_from_string(json_string: str) -> Optional[Dict[str, Any]]:
    """Extract JSON data from a string that contains a JSON code block."""
    clean_json = re.sub(r'^```json\s*', '', json_string.strip())
    clean_json = re.sub(r'\s*```$', '', clean_json)

    try:
        return json.loads(clean_json)
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        print(f"Problematic string: {json_string[:100]}...")
        return None

def load_results(file_path: str) -> List[Dict[str, Any]]:
    """Load results from a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)

        results = []
        if isinstance(raw_data, list):
            for item in raw_data:
                if isinstance(item, str):
                    extracted_data = extract_json_from_string(item)
                    if extracted_data:
                        results.append(extracted_data)
                elif isinstance(item, dict):
                    results.append(item)
            return results
        else:
            print(f"Warning: Expected a list, got {type(raw_data)}. Wrapping in list.")
            return [raw_data]

    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return []

def sort_by_citizen_science_score(results: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Sort results by citizen science score in descending order."""
    return sorted(results, key=lambda x: x.get('is_citizen_science', 0), reverse=True)

def save_sorted_results(results: List[Dict[str, Any]], output_path: str):
    """Save sorted results to a CSV file."""
    df = pd.DataFrame(results)
    df.to_csv(output_path, index=False, encoding='utf-8')
    print(f"Sorted results saved to: {output_path}")

def display_score_summary(results: List[Dict[str, Any]]):
    """Display a summary of the score distribution."""
    print("\n" + "="*50)
    print("CITIZEN SCIENCE SCORE DISTRIBUTION")
    print("="*50)

    # Count by score
    score_counts = {}
    for result in results:
        score = result.get('is_citizen_science', 0)
        score_counts[score] = score_counts.get(score, 0) + 1

    # Display distribution
    for score in sorted(score_counts.keys(), reverse=True):
        count = score_counts[score]
        percentage = (count / len(results)) * 100 if results else 0
        print(f"Score {score}: {count:4d} projects ({percentage:5.1f}%)")

    print(f"\nTotal projects: {len(results)}")

    # Show top 10 projects
    top_projects = results[:10]
    if top_projects:
        print(f"\nTOP 10 CITIZEN SCIENCE PROJECTS:")
        print("-" * 60)
        for i, project in enumerate(top_projects, 1):
            score = project.get('is_citizen_science', 0)
            name = project.get('project_name', 'Unknown')
            print(f"{i:2d}. [{score}] {name}")

# ====================================================
# CONFIGURATION - MODIFY THIS SECTION
# ====================================================

# SET YOUR INPUT FILE PATH HERE
# Option 1: Absolute path
# input_file = '/home/marcello/projects/csnl/CS.nl/data/processed/your_results.json'

# Option 2: Relative path (relative to notebook location)
input_file = '/content/second_filtration.json'

# Optional: Set custom output path (if not specified, it will be auto-generated)
output_file = '/content/sorted_second_filtration.csv'  # Set to None for auto-generation, or specify a path like './outputs/sorted_results.csv'

# ====================================================
# MAIN EXECUTION
# ====================================================

print("CITIZEN SCIENCE PROJECT SORTING ANALYSIS")
print("="*50)
print(f"Input file: {input_file}")
print(f"Output file: {output_file if output_file else 'Will be auto-generated'}")
print()

# Check if input file exists
if not Path(input_file).exists():
    print(f"❌ Error: Input file '{input_file}' not found!")
    print(f"Please check the path and make sure the file exists.")
    print(f"Current working directory: {Path.cwd()}")
    print(f"Available files in current directory: {list(Path('.').glob('*.json'))}")
else:
    print(f"✅ Input file found: {input_file}")

    # Load results from the JSON file
    print("Loading results...")
    results = load_results(str(input_file))

    if not results:
        print("❌ No results found in the input file!")
    else:
        print(f"✅ Loaded {len(results)} projects")

        # Sort by citizen science score
        print("Sorting by citizen science score...")
        sorted_results = sort_by_citizen_science_score(results)
        print(f"✅ Sorting completed!")

        # Determine output path
        if output_file:
            output_path = output_file
        else:
            input_path = Path(input_file)
            output_path = input_path.parent / f"{input_path.stem}_sorted.csv"

        print(f"Saving results to: {output_path}")

        # Save sorted results to CSV
        save_sorted_results(sorted_results, str(output_path))

        # Display summary
        display_score_summary(sorted_results)

        # Optional: Create DataFrame for further analysis
        df = pd.DataFrame(sorted_results)
        print("\n" + "="*50)
        print("DATASET OVERVIEW")
        print("="*50)
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")

        # Show score statistics if available
        if 'is_citizen_science' in df.columns:
            print(f"\nScore Statistics:")
            print(f"Mean score: {df['is_citizen_science'].mean():.2f}")
            print(f"Median score: {df['is_citizen_science'].median():.2f}")
            print(f"Max score: {df['is_citizen_science'].max()}")
            print(f"Min score: {df['is_citizen_science'].min()}")

        print(f"\n🎉 Analysis completed successfully!")
        print(f"📊 Sorted {len(sorted_results)} projects by citizen science score")
        print(f"💾 Results saved to: {output_path}")

        # Display first few rows
        print("\nFirst 5 rows of sorted data:")
        display(df.head())

        # Make the DataFrame available for further use
        globals()['sorted_df'] = df
        globals()['sorted_results'] = sorted_results

        print("\n📌 Variables created:")
        print("   - sorted_df: DataFrame with sorted results")
        print("   - sorted_results: List of sorted dictionaries")

# License

Copyright(C) 2025 Sadra & Marcello

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY PURPOSE. See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program. If not, see

This program is licensed under the [GNU General Public License v3.0](https://www.gnu.org/licenses/gpl-3.0.en.html).